# 13 · Vize Ekleri (22 Ekim danışman görüşmesi)

| Danışmanın isteği | Bölüm |
|---|---|
| Veri seti tablosu: sınıf başına eğitim / doğrulama / test sayıları | A |
| Test setinin **tamamında** sonuç: kaç belge verildi, kaçını OCR okuyamadı, okunanlardan kaçı doğru/yanlış | B |
| Şablon **adım adım**: "sadece university → şu; university + research → şu; + yıl → şu" | C |
| Sözlük: nereden alındı, kaç kelime; ham → genel sözlük → **iki aşamalı sözlük (genel + alan)**; her aşamanın sonucu | D |
| Her sınıftan bir örnek: OCR ham metni → genel sözlük → iki aşamalı sözlük | E |

Hiçbir yeni OCR yapılmaz; 03'ün temizlenmiş metni (`cleaned_dataset.csv`) ve 04'ün modeli kullanılır. Süre ~10 dk.

In [ ]:
!pip -q install symspellpy

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys, json, importlib, time, textwrap
from collections import Counter
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

ROOT = Path("/content/drive/MyDrive/bitirme_ocr")
DATA, MODELS = ROOT / "data", ROOT / "models"
FIG = ROOT / "figures"; FIG.mkdir(exist_ok=True)
OUT = ROOT / "vize"; OUT.mkdir(exist_ok=True)
CLASSES = json.loads((ROOT / "siniflar.json").read_text())["CLASSES"]
TR = {"email": "E-posta", "invoice": "Fatura", "resume": "CV", "scientific_publication": "Bilimsel makale"}
SEED = 42
print(CLASSES)

In [ ]:
%%writefile /content/drive/MyDrive/bitirme_ocr/text_clean.py
import re
import unicodedata

# OCR'ın sık ürettiği tipografik karakterleri sadeleştir
_CHAR_MAP = str.maketrans({
    "‘": "'", "’": "'", "“": '"', "”": '"',
    "–": "-", "—": "-", "•": "-", "·": "-",
    " ": " ", "­": "",
})
_WORDISH = re.compile(r"[A-Za-z0-9]")
_NOISE_TOKEN = re.compile(r"^[^\w$%@&#+]+$")          # sadece sembol: | ~ ` ' . , _ vb.
_EDGE_JUNK_L = re.compile(r"^[|_~`'\"\s]+")
_EDGE_JUNK_R = re.compile(r"[|_~`\s]+$")
_EMAIL = re.compile(r"\b[\w.+-]+@[\w-]+(?:\.[\w-]+)+\b")
_URL = re.compile(r"\b(?:https?://|www\.)\S+", re.I)
_MONEY = re.compile(r"[$€£]\s?\d[\d,]*(?:\.\d+)?")
_NUM = re.compile(r"\d+")


def alnum_ratio(s):
    s = s.replace(" ", "")
    return sum(c.isalnum() for c in s) / len(s) if s else 0.0


def clean_text(raw, min_line_alnum=0.4):
    """OCR metnini temizler; büyük/küçük harf, e-posta, tutar gibi bilgiler korunur
    (bilgi çıkarımı aşamasında gerekecek)."""
    if not isinstance(raw, str) or not raw.strip():
        return ""
    t = unicodedata.normalize("NFKC", raw).translate(_CHAR_MAP)
    t = "".join(c for c in t if c == "\n" or c.isprintable())
    t = re.sub(r"(\w)-\n(\w)", r"\1\2", t)               # satır sonu tirelemesini birleştir
    lines = []
    for line in t.split("\n"):
        tokens = [tok for tok in line.split() if not _NOISE_TOKEN.match(tok)]
        line = " ".join(tokens)
        line = _EDGE_JUNK_R.sub("", _EDGE_JUNK_L.sub("", line))
        if len(line) < 2 or not _WORDISH.search(line):
            continue
        if alnum_ratio(line) < min_line_alnum:              # çoğu sembol olan çöp satır
            continue
        lines.append(line)
    return "\n".join(lines)


def normalize_for_model(clean):
    """Sınıflandırıcı için: küçük harf + e-posta/URL/tutar/sayıları ortak simgeye çevir.
    Böylece model 'john@acme.com' yerine 'bir e-posta adresi var' bilgisini öğrenir."""
    t = clean.lower()
    t = _EMAIL.sub(" emailaddr ", t)
    t = _URL.sub(" urladdr ", t)
    t = _MONEY.sub(" moneyamt ", t)
    t = _NUM.sub("0", t)
    t = re.sub(r"[^a-z0\s]", " ", t)
    return " ".join(t.split())


def garbage_ratio(raw):
    """Ham OCR çıktısındaki anlamsız token oranı (harf/rakam içermeyen ya da
    harf-rakam-sembol karışık kısa parçalar)."""
    toks = raw.split() if isinstance(raw, str) else []
    if not toks:
        return 1.0
    bad = 0
    for tok in toks:
        if not _WORDISH.search(tok):
            bad += 1
        elif len(tok) <= 3 and alnum_ratio(tok) < 0.67:
            bad += 1
    return bad / len(toks)


In [ ]:
%%writefile /content/drive/MyDrive/bitirme_ocr/sablon.py
"""Belge türü şablonları (danışman isteği, 15 Ekim): her türün hemen her belgesinde bulunan ortak ifadeler.

OCR metninde bir türün şablon ifadeleri bulunuyorsa bu, makine öğrenmesi modelinin kararını destekleyen
ikinci bir kanıttır (ör. faturada "invoice", "total", para tutarı; mektupta "Dear", "Sincerely").
Veri seti İngilizce olduğu için ifadeler İngilizcedir; Türkçe karşılıkları yorumda verilmiştir.
"""
import re

# her ifade: (kısa ad, düzenli ifade) · metin küçük harfe çevrilmiş temiz OCR metnine uygulanır
TEMPLATES = {
    "email": [  # e-posta başlık alanları
        ("from:", r"^\s*from\s*[:;]"),
        ("to:", r"^\s*to\s*[:;]"),
        ("subject:", r"^\s*(?:subject|subj)\s*[:;]"),
        ("sent/date:", r"^\s*(?:sent|date)\s*[:;]"),
        ("cc:", r"^\s*b?cc\s*[:;]"),
        ("@ adresi", r"[\w.+-]+@[\w-]+"),
        ("message", r"\bmessage\b"),
    ],
    "invoice": [  # fatura: numara, tutar, miktar, ödeme
        ("invoice", r"\binvoice\b"),
        ("total", r"\btotal\b"),
        ("amount", r"\bamount\b"),
        ("para tutarı", r"[$€£]\s?\d|\b\d{1,3}(?:,\d{3})*\.\d{2}\b"),
        ("qty/quantity", r"\b(?:qty|quantity)\b"),
        ("price", r"\bprice\b"),
        ("bill/ship/sold to", r"\b(?:bill|ship|sold|remit)\s*to\b"),
        ("payment/due", r"\b(?:payment|payable|due|terms)\b"),
    ],
    "resume": [  # CV bölüm başlıkları
        ("education", r"\beducation\b"),
        ("experience", r"\b(?:experience|employment)\b"),
        ("university/college", r"\b(?:university|college|institute)\b"),
        ("degree", r"\b(?:b\.?s\.?|m\.?s\.?|ph\.?d\.?|bachelor|master|doctor|degree)\b"),
        ("born/birth", r"\b(?:born|birth|married)\b"),
        ("research", r"\bresearch\b"),
        ("publications/references", r"\b(?:publications|references|honors|awards)\b"),
        # 21 Ekim: CV'yi bilimsel makaleden ayıran, CV'ye özgü ifadeler eklendi (university korunuyor)
        ("biographical sketch/cv", r"\b(?:biographical\s+sketch|curriculum\s+vitae|resume)\b"),
        ("position title", r"\bposition\s+title\b"),
        ("professor", r"\b(?:assistant|associate|full|visiting)?\s*professor\b"),
        ("yıl aralığı", r"\b(?:19|20)\d{2}\s*[-–]\s*(?:(?:19|20)?\d{2}\b|present)"),
        ("present", r"\bpresent\b"),
    ],
    "form": [  # form: faks kapağı, doldurulacak alanlar
        ("fax/facsimile", r"\b(?:fax|facsimile|telecopy|telecopier)\b"),
        ("transmission", r"\btransmi(?:ssion|tted|t)\b"),
        ("pages", r"\bpages?\b"),
        ("name:", r"\bname\s*[:;]"),
        ("date:", r"\bdate\s*[:;]"),
        ("signature/approved", r"\b(?:signature|signed|approv(?:ed|al))\b"),
        ("request/number", r"\b(?:request|number|no\.)\b"),
    ],
    "letter": [  # mektup / resmi yazı (dilekçe): hitap, kapanış, ek
        ("dear", r"^\s*dear\b"),
        ("sincerely/regards", r"\b(?:sincerely|yours\s+truly|very\s+truly|regards|respectfully|cordially)\b"),
        ("re:", r"^\s*re\s*[:;]"),
        ("enclosed/enclosure", r"\benclos(?:ed|ure)\b"),
        ("thank you", r"\bthank\s+you\b"),
        ("letter", r"\bletter\b"),
        ("please", r"\bplease\b"),
    ],
    "news_article": [  # gazete haberi
        ("said", r"\bsaid\b"),
        ("ajans (AP/UPI/Reuters)", r"\((?:ap|upi)\)|\breuters\b|\bassociated press\b"),
        ("percent", r"\bpercent\b"),
        ("according to", r"\baccording to\b"),
        ("spokesman/officials", r"\b(?:spokesman|spokeswoman|officials?)\b"),
        ("yesterday/today", r"\b(?:yesterday|today|last week)\b"),
        ("times/post/journal", r"\b(?:times|post|journal|tribune|herald|news)\b"),
    ],
    "scientific_publication": [  # bilimsel makale bölümleri
        ("abstract", r"\babstract\b"),
        ("introduction", r"\bintroduction\b"),
        ("methods/results", r"\b(?:methods?|results)\b"),
        ("references", r"\breferences\b"),
        ("et al.", r"\bet\s+al\b"),
        ("fig./table", r"\b(?:fig|figure|table)\b"),
        ("journal/vol.", r"\b(?:journal|vol|volume)\b"),
    ],
}

# İlk sürüm (20 Ekim): karşılaştırma için saklanır. CV şablonunda sadece ilk 7 ifade vardı.
_NEW_RESUME = {"biographical sketch/cv", "position title", "professor", "yıl aralığı", "present"}
TEMPLATES_V1 = {c: [(n, p) for n, p in pats if not (c == "resume" and n in _NEW_RESUME)]
                for c, pats in TEMPLATES.items()}

_CACHE = {}


def _compiled(templates):
    key = id(templates)
    if key not in _CACHE:
        _CACHE[key] = {c: [(n, re.compile(p, re.I | re.M)) for n, p in pats] for c, pats in templates.items()}
    return _CACHE[key]


_COMPILED = _compiled(TEMPLATES)


def match_template(text, classes=None, templates=None):
    """{sınıf: [eşleşen ifade adları]}"""
    comp = _compiled(templates or TEMPLATES)
    text = text or ""
    return {c: [n for n, rx in comp[c] if rx.search(text)] for c in (classes or comp)}


def fit_weights(texts, labels, classes, templates=None):
    """Her ifadeye veriden ağırlık: (kendi sınıfında geçtiği belge oranı) − (diğer sınıflardaki en yüksek oran).

    Örnek: "education" CV'lerin %59'unda, makalelerin %1'inde geçer → ağırlık ≈ 0,58.
    "university" CV'lerin %93'ünde, makalelerin %36'sında geçer → ağırlık ≈ 0,57'den düşük kalır ve
    tek başına makaleyi CV'ye çeviremez. Negatif çıkanlar küçük bir pozitif değere (0,01) çekilir.
    """
    templates = templates or TEMPLATES
    labels = list(labels)
    matches = [match_template(t, list(templates), templates) for t in texts]
    rate = {}
    for c in templates:
        for n, _ in templates[c]:
            rate[(c, n)] = {k: (sum(1 for m, l in zip(matches, labels) if l == k and n in m[c]) /
                                max(1, sum(1 for l in labels if l == k))) for k in classes}
    weights = {}
    for c in classes:
        weights[c] = {}
        for n, _ in templates[c]:
            r = rate[(c, n)]
            others = max((r[k] for k in classes if k != c), default=0.0)
            weights[c][n] = round(max(0.01, r[c] - others), 4)
    return weights


def template_scores(text, classes=None, weights=None, templates=None):
    """{sınıf: şablon puanı (0-1)}. weights verilmezse her ifade eşit sayılır (eşleşen ifade oranı)."""
    templates = templates or TEMPLATES
    m = match_template(text, classes, templates)
    if weights is None:
        return {c: len(v) / len(templates[c]) for c, v in m.items()}
    return {c: sum(weights[c].get(n, 0) for n in v) / max(1e-9, sum(weights[c].values())) for c, v in m.items()}


def template_vector(text, classes):
    """Model için özellik vektörü: her ifade için 0/1 + her sınıf için eşleşme oranı."""
    m = match_template(text, classes)
    bits = [float(n in m[c]) for c in classes for n, _ in _COMPILED[c]]
    return bits + [len(m[c]) / len(TEMPLATES[c]) for c in classes]


def template_feature_names(classes):
    return [f"{c}:{n}" for c in classes for n, _ in _COMPILED[c]] + [f"{c}:oran" for c in classes]


def rule_predict(text, classes, weights=None, templates=None):
    """Sadece şablona göre tahmin: en yüksek puanlı sınıf (hiç eşleşme yoksa None)."""
    s = template_scores(text, classes, weights, templates)
    best = max(s, key=s.get)
    return best if s[best] > 0 else None


In [ ]:
%%writefile /content/drive/MyDrive/bitirme_ocr/ocr_fix.py
"""Sözlük tabanlı OCR hata düzeltme (danışman isteği, 15 Ekim: "Dater" → "Date" gibi hatalar).

Üç adım, sırayla:
1. OCR karışıklık kuralları: Tesseract'ın sık karıştırdığı harf grupları (rn↔m, 1↔l, 0↔o, vv↔w, cl↔d ...).
   Kelime sözlükte yoksa ve bir kural uygulanınca sözlükte bir kelimeye dönüşüyorsa o kelime alınır.
2. Alan adı kuralı: satır başındaki kelime bilinen bir alan adına (Date, From, To, Total ...) 1 harf
   uzaklıktaysa düzeltilir. "Dater" sözlükte geçerli bir kelime olduğu için sadece bu kural yakalayabilir.
3. SymSpell: sözlükte olmayan kelime, sıklık sözlüğündeki en yakın kelimeye çevrilir
   (5 harfe kadar en fazla 1, daha uzunlarda en fazla 2 harf farkı).

Rakam, sembol içeren kelimeler ve 3 harften kısa kelimeler değiştirilmez.
Kişi/firma adları da sözlükte olmadığı için yanlış "düzeltilebilir"; bu yüzden düzeltilmiş metin
sadece sınıflandırma için kullanılır, içerik çıkarımı (ad, firma) orijinal metinden yapılır.
"""
import re
from collections import Counter
from importlib import resources

from symspellpy import SymSpell, Verbosity

_SYM = None
_WORDS = None
OCR_CONFUSIONS = [("rn", "m"), ("m", "rn"), ("1", "l"), ("l", "1"), ("0", "o"), ("vv", "w"), ("cl", "d"),
                  ("ii", "u"), ("li", "h"), ("5", "s"), ("8", "b"), ("|", "l"), ("l", "i"), ("i", "l")]
FIELD_NAMES = ["date", "from", "to", "subject", "name", "address", "total", "amount", "invoice", "phone",
               "fax", "pages", "re", "cc", "title", "company", "signature"]
_TOKEN = re.compile(r"[A-Za-z0-9|]+")


def _load():
    global _SYM, _WORDS
    if _SYM is None:
        _SYM = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)
        path = resources.files("symspellpy") / "frequency_dictionary_en_82_765.txt"
        _SYM.load_dictionary(str(path), term_index=0, count_index=1)
        _WORDS = _SYM.words
    return _SYM


def in_dictionary(word):
    _load()
    return word.lower() in _WORDS


def _match_case(src, dst):
    if src.isupper():
        return dst.upper()
    if src[:1].isupper():
        return dst[:1].upper() + dst[1:]
    return dst


def _confusion_fix(low):
    for a, b in OCR_CONFUSIONS:
        if a in low:
            cand = low.replace(a, b)
            if cand in _WORDS:
                return cand
    return None


def correct_word(word, line_start=False):
    """(düzeltilmiş kelime, kural adı) — değişiklik yoksa (word, None)."""
    sym = _load()
    low = word.lower()
    if line_start:
        for f in FIELD_NAMES:
            if len(f) >= 3 and low != f and len(low) == len(f) + 1 and low.startswith(f):
                return _match_case(word, f), "alan adı"
    if len(low) < 3 or low in _WORDS:
        return word, None
    fixed = _confusion_fix(low)
    if fixed:
        return _match_case(word, fixed), "OCR karışıklığı"
    if not low.isalpha():
        return word, None
    max_d = 1 if len(low) <= 5 else 2
    sug = sym.lookup(low, Verbosity.TOP, max_edit_distance=max_d)
    if sug and sug[0].term != low:
        return _match_case(word, sug[0].term), "SymSpell"
    return word, None


def correct_text(text, log=None):
    """Metindeki kelimeleri düzeltir. `log` bir Counter ise (orijinal, düzeltme, kural) sayılır."""
    if not text:
        return text
    out_lines = []
    for line in text.split("\n"):
        pieces, last, first = [], 0, True
        for m in _TOKEN.finditer(line):
            w = m.group(0)
            new, rule = correct_word(w, line_start=first and m.start() <= 2)
            first = False
            pieces.append(line[last:m.start()])
            pieces.append(new)
            last = m.end()
            if rule and log is not None:
                log[(w, new, rule)] += 1
        pieces.append(line[last:])
        out_lines.append("".join(pieces))
    return "\n".join(out_lines)


def dict_ratio(text):
    """Harflerden oluşan (≥3) kelimelerin sözlükte bulunma oranı."""
    _load()
    toks = [t.lower() for t in re.findall(r"[A-Za-z]+", text or "") if len(t) >= 3]
    return sum(t in _WORDS for t in toks) / len(toks) if toks else 0.0


# ---------------------------------------------------------------- iki aşamalı sözlük (22 Ekim danışman isteği)
# Genel İngilizce sözlük bilimsel terimleri tanımadığı için onları "düzeltiyordu" (DNA → DAN, tumor → tutor).
# Çözüm: ikinci bir ALAN SÖZLÜĞÜ. Train belgelerinde sık geçen ama genel sözlükte olmayan, tek bir belge
# türünde yoğunlaşan kelimeler (DNA, PhD, NIH, tumor ...) alan sözlüğüne alınır.
#   1. aşama: kelime alan sözlüğündeyse dokunulmaz (korunur)
#   2. aşama: alan kelimesine 1 harf uzaklıktaysa ona düzeltilir (ör. "DNAs" değil, "tumar" → "tumor")
#   3. kalanlar genel sözlükle düzeltilir (eski yöntem)
# Yaygın OCR hataları ("wih", "fom") her belge türünde geçtiği için yoğunlaşma şartını geçemez, alan sözlüğüne girmez.

def build_domain_dictionary(texts, labels, min_docs=5, min_share=0.6):
    """Alan sözlüğü: {kelime: belge sayısı}. Sadece train metinleri verilmelidir.

    Şartlar: (1) genel sözlükte yok, (2) en az `min_docs` belgede geçiyor,
    (3) geçtiği belgelerin en az `min_share` kadarı tek bir türde (alan terimi; OCR hatası değil).
    """
    _load()
    per_word = {}
    for t, lab in zip(texts, labels):
        for w in {w.lower() for w in re.findall(r"[A-Za-z]+", t or "") if len(w) >= 3}:
            if w not in _WORDS:
                per_word.setdefault(w, Counter())[lab] += 1
    out = {}
    for w, c in per_word.items():
        n = sum(c.values())
        if n >= min_docs and max(c.values()) / n >= min_share:
            out[w] = n
    return out


class TwoStageFixer:
    """Genel sözlük + alan sözlüğü ile düzeltme. `correct_text` ile aynı arayüz."""

    def __init__(self, domain):
        _load()
        self.domain = dict(domain)
        self.dsym = SymSpell(max_dictionary_edit_distance=1, prefix_length=7)
        for w, c in self.domain.items():
            self.dsym.create_dictionary_entry(w, c)

    def correct_word(self, word, line_start=False):
        low = word.lower()
        if low in self.domain:
            return word, None                                    # alan kelimesi korunur
        if len(low) >= 5 and low.isalpha() and low not in _WORDS:
            sug = self.dsym.lookup(low, Verbosity.TOP, max_edit_distance=1)
            if sug and sug[0].term != low:
                return _match_case(word, sug[0].term), "alan sözlüğü"
        return correct_word(word, line_start)

    def correct_text(self, text, log=None):
        if not text:
            return text
        out_lines = []
        for line in text.split("\n"):
            pieces, last, first = [], 0, True
            for m in _TOKEN.finditer(line):
                w = m.group(0)
                new, rule = self.correct_word(w, line_start=first and m.start() <= 2)
                first = False
                pieces.append(line[last:m.start()])
                pieces.append(new)
                last = m.end()
                if rule and log is not None:
                    log[(w, new, rule)] += 1
            pieces.append(line[last:])
            out_lines.append("".join(pieces))
        return "\n".join(out_lines)

    def valid_ratio(self, text):
        """Genel ya da alan sözlüğünde bulunan kelime oranı."""
        toks = [t.lower() for t in re.findall(r"[A-Za-z]+", text or "") if len(t) >= 3]
        return sum(t in _WORDS or t in self.domain for t in toks) / len(toks) if toks else 0.0


def dictionary_info():
    """Genel sözlüğün kaynağı ve boyutu (sunum için)."""
    _load()
    return {"dosya": "frequency_dictionary_en_82_765.txt (symspellpy paketi)",
            "kelime sayısı": len(_WORDS),
            "kaynak": "SymSpell; SCOWL kelime listesi ile Google Books Ngram sıklıklarının kesişimi"}


In [ ]:
sys.path.insert(0, str(ROOT))
import text_clean, sablon, ocr_fix
for m in (text_clean, sablon, ocr_fix): importlib.reload(m)
from text_clean import normalize_for_model

df = pd.read_csv(DATA / "cleaned_dataset.csv")
df = df[df.label.isin(CLASSES)].reset_index(drop=True)
df["text_clean"] = df["text_clean"].fillna(""); df["text_norm"] = df["text_norm"].fillna("")
train = df[(df.split == "train") & ~df.is_empty]
val, test = df[df.split == "validation"], df[df.split == "test"]
print(len(df), len(train), len(val), len(test))

## A · Veri seti tablosu

In [ ]:
tab = pd.crosstab(df.label, df.split).reindex(CLASSES)[["train", "validation", "test"]]
tab["toplam"] = tab.sum(axis=1)
tab["OCR'ın okuyamadığı (< 5 kelime)"] = df.groupby("label")["is_empty"].sum().reindex(CLASSES)
tab.loc["TOPLAM"] = tab.sum()
tab = tab.rename(index=TR, columns={"train": "eğitim", "validation": "doğrulama"})
tab.to_csv(OUT / "veri_seti_tablosu.csv")
tab

## B · Test setinin tamamı: OCR hatası ve sınıflandırma ayrı ayrı

Danışmanın istediği biçim: "200 belge verdim, OCR 180'ini okuyabildi, 20'si OCR hatası; okunan 180'in 150'si doğru, 30'u yanlış."
OCR'ın okuyamadığı belge = temizlikten sonra 5 kelimeden az metin kalan belge. Model: 04'ün en iyi modeli (vize modeli).

In [ ]:
import joblib
from sklearn.metrics import accuracy_score
model04 = joblib.load(MODELS / "best_model.joblib")
test = test.copy()
test["tahmin"] = model04.predict(test.text_norm)
test["dogru"] = test.tahmin == test.label
rows = []
for c in CLASSES + ["TOPLAM"]:
    t = test if c == "TOPLAM" else test[test.label == c]
    ok = t[~t.is_empty]
    rows.append({"sınıf": TR.get(c, c), "test belgesi": len(t), "OCR okuyamadı": int(t.is_empty.sum()),
                 "OCR okudu": len(ok), "doğru": int(ok.dogru.sum()), "yanlış": int((~ok.dogru).sum()),
                 "başarı, okunanlarda (%)": round(100 * ok.dogru.mean(), 1),
                 "başarı, tüm test (%)": round(100 * t.dogru.mean(), 1)})
full = pd.DataFrame(rows)
full.to_csv(OUT / "test_tamami_ocr_ve_siniflandirma.csv", index=False)
full

## C · Şablon adım adım (CV şablonu)

Diğer türlerin şablonu sabit tutulur; CV şablonuna ifadeler **birer birer** eklenir ve sadece şablonla (makine öğrenmesi olmadan) test seti sınıflandırılır.
Her satır bir önceki satıra bir ifade eklenmiş halidir. Son satırda ifadeler veriden ağırlık alır (07'deki iyileştirme).

In [ ]:
from sablon import TEMPLATES, rule_predict, fit_weights
from sklearn.metrics import f1_score, precision_recall_fscore_support

ORDER = ["university/college", "research", "yıl aralığı", "education", "experience", "degree",
         "publications/references", "born/birth", "biographical sketch/cv", "professor", "position title", "present"]
PAT = dict(TEMPLATES["resume"])
test_low = test.text_clean.str.lower()

def evaluate_rule(templates, weights=None):
    pred = test_low.map(lambda t: rule_predict(t, CLASSES, weights=weights, templates=templates)).fillna("eşleşme yok").values
    y = test.label.values
    _, rec, f, _ = precision_recall_fscore_support(y, pred, labels=CLASSES, zero_division=0)
    return {"accuracy (%)": round(100 * accuracy_score(y, pred), 1), "macro-F1": round(f1_score(y, pred, labels=CLASSES, average="macro"), 3),
            "CV F1": round(f[CLASSES.index("resume")], 3), "makale F1": round(f[CLASSES.index("scientific_publication")], 3),
            "makale → CV hatası": int(((y == "scientific_publication") & (pred == "resume")).sum()),
            "CV bulunamadı": int(((y == "resume") & (pred != "resume")).sum())}

steps = []
for k in range(1, len(ORDER) + 1):
    tpl = {c: (v if c != "resume" else [(n, PAT[n]) for n in ORDER[:k]]) for c, v in TEMPLATES.items()}
    steps.append({"adım": k, "CV şablonu": " + ".join(ORDER[:k]) if k <= 3 else f"+ {ORDER[k - 1]}", **evaluate_rule(tpl)})
W8 = fit_weights(train.text_clean.str.lower(), train.label, CLASSES)
steps.append({"adım": len(ORDER) + 1, "CV şablonu": "tüm ifadeler + veriden ağırlık", **evaluate_rule(TEMPLATES, W8)})
abl = pd.DataFrame(steps)
abl.to_csv(OUT / "sablon_adim_adim.csv", index=False)
abl

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.2))
x = np.arange(len(abl)); labs = [f"{r.adım}" for r in abl.itertuples()]
ax.plot(x, abl["accuracy (%)"], "o-", color="#0f6e66", label="accuracy (%)")
ax.plot(x, 100 * abl["CV F1"], "s--", color="#e0a458", label="CV F1 (×100)")
ax2 = ax.twinx(); ax2.bar(x, abl["makale → CV hatası"], alpha=.25, color="#b03a2e", label="makale → CV hatası")
ax.set_xticks(x); ax.set_xticklabels([("1 university", "2 + research", "3 + yıl")[i] if i < 3 else s for i, s in enumerate(labs)], rotation=30, ha="right")
ax.set_ylabel("%"); ax2.set_ylabel("makale → CV (belge)"); ax.grid(alpha=.3)
ax.set_title("CV şablonuna ifadeler birer birer eklenince sadece şablonla sınıflandırma")
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels(); ax.legend(h1 + h2, l1 + l2, loc="lower right", fontsize=8)
plt.tight_layout(); plt.savefig(OUT / "sablon_adim_adim.png", dpi=150); plt.show()

## D · Sözlük aşamaları

| Aşama | Ne yapılır |
|---|---|
| 1 · Ham OCR | Düzeltme yok |
| 2 · Genel sözlük | SymSpell İngilizce sıklık sözlüğü + OCR karışıklık kuralları + alan adı kuralı (07'deki yöntem) |
| 3 · İki aşamalı sözlük | Önce **alan sözlüğü** (train belgelerinde sık geçen, genel sözlükte olmayan, tek türde yoğunlaşan kelimeler: DNA, PhD …) korunur/düzeltilir, sonra genel sözlük |

Alan sözlüğü **sadece train** belgelerinden çıkarılır.

In [ ]:
from ocr_fix import correct_text, dictionary_info, build_domain_dictionary, TwoStageFixer
info = dictionary_info(); print(info)
domain = build_domain_dictionary(train.text_clean, train.label, min_docs=5, min_share=0.6)
print("Alan sözlüğü:", len(domain), "kelime")
dom_tab = pd.Series(domain).sort_values(ascending=False)
print(", ".join(dom_tab.index[:60]))
dom_tab.rename("train belge sayısı").to_csv(OUT / "alan_sozlugu.csv")

In [ ]:
log_g, log_2 = Counter(), Counter()
fixer = TwoStageFixer(domain)
t0 = time.time()
df["text_gen"] = [correct_text(t, log_g) for t in df.text_clean]
df["text_two"] = [fixer.correct_text(t, log_2) for t in df.text_clean]
print(f"{time.time() - t0:.0f} sn")
for col in ["text_gen", "text_two"]:
    df[col + "_norm"] = df[col].map(normalize_for_model)

# Genel sözlüğün bozduğu alan kelimeleri (DNA → DAN gibi): orijinali alan sözlüğünde olan düzeltmeler
broken = Counter({k: v for k, v in log_g.items() if k[0].lower() in domain})
print("Genel sözlüğün bozduğu alan kelimesi:", sum(broken.values()), "· en sık:", broken.most_common(10))
print("İki aşamalıda bozulan alan kelimesi:", sum(v for k, v in log_2.items() if k[0].lower() in domain))
rules2 = Counter()
for (o, n, r), v in log_2.items(): rules2[r] += v
rulesg = Counter()
for (o, n, r), v in log_g.items(): rulesg[r] += v
print("Genel:", dict(rulesg), "· İki aşamalı:", dict(rules2))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

def word_tfidf():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.9, sublinear_tf=True, stop_words="english",
                           token_pattern=r"(?u)\b[a-z0][a-z0]+\b")
def char_tfidf():
    return TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.9, sublinear_tf=True, max_features=100_000)

def fit_eval(col, make):
    tr, va, te = df.loc[train.index], df.loc[val.index], df.loc[test.index]
    v = make().fit(tr[col]); best = None
    for C in (0.05, 0.1, 0.5, 1, 5):
        clf = LinearSVC(C=C, random_state=SEED).fit(v.transform(tr[col]), tr.label)
        f = f1_score(va.label, clf.predict(v.transform(va[col])), average="macro")
        if best is None or f > best[0]: best = (f, clf)
    return accuracy_score(te.label, best[1].predict(v.transform(te[col])))

stage_rows = []
for name, col, ratio in [("1 · Ham OCR", "text_norm", lambda t: fixer.valid_ratio(t)),
                         ("2 · Genel sözlük", "text_gen_norm", lambda t: fixer.valid_ratio(t)),
                         ("3 · İki aşamalı sözlük (genel + alan)", "text_two_norm", lambda t: fixer.valid_ratio(t))]:
    src = {"text_norm": "text_clean", "text_gen_norm": "text_gen", "text_two_norm": "text_two"}[col]
    r = df[src].map(ratio)
    stage_rows.append({"aşama": name,
                       **{f"geçerli kelime, {TR[c]} (%)": round(100 * r[df.label == c].mean(), 1) for c in CLASSES},
                       "geçerli kelime, ortalama (%)": round(100 * r.mean(), 1),
                       "düzeltme sayısı": {"text_norm": 0, "text_gen_norm": sum(log_g.values()), "text_two_norm": sum(log_2.values())}[col],
                       "bozulan alan kelimesi": {"text_norm": 0, "text_gen_norm": sum(broken.values()),
                                                 "text_two_norm": sum(v for k, v in log_2.items() if k[0].lower() in domain)}[col],
                       "test accuracy, kelime TF-IDF (%)": round(100 * fit_eval(col, word_tfidf), 2),
                       "test accuracy, karakter TF-IDF (%)": round(100 * fit_eval(col, char_tfidf), 2)})
stages = pd.DataFrame(stage_rows)
stages.to_csv(OUT / "sozluk_asamalari.csv", index=False)
stages.T

In [ ]:
ex = pd.DataFrame([{"orijinal": o, "genel sözlük": n, "adet": v} for (o, n, r), v in broken.most_common(15)],
                  columns=["orijinal", "genel sözlük", "adet"])
ex["iki aşamalı"] = ex["orijinal"]
ex.to_csv(OUT / "sozluk_bozulan_alan_kelimeleri.csv", index=False)
ex

## E · Her sınıftan bir örnek: ham OCR → genel sözlük → iki aşamalı sözlük

In [ ]:
def changed_lines(r, k=6):
    out = []
    for a, b, c in zip(r.text_clean.splitlines(), r.text_gen.splitlines(), r.text_two.splitlines()):
        if a != b or a != c:
            out.append((a[:70], b[:70], c[:70]))
    return out[:k]

picked = {}
for c in CLASSES:
    pool = df[(df.label == c) & (df.split == "test") & ~df.is_empty].copy()
    pool["n_gen"] = [sum(a != b for a, b in zip(x.splitlines(), y.splitlines())) for x, y in zip(pool.text_clean, pool.text_gen)]
    pool["n_dom"] = [sum(a != b for a, b in zip(x.splitlines(), y.splitlines())) for x, y in zip(pool.text_gen, pool.text_two)]
    pool = pool[pool.n_gen >= 3].sort_values(["n_dom", "n_gen"], ascending=False)
    picked[c] = pool.iloc[0] if len(pool) else None

rows = []
for c, r in picked.items():
    if r is None: continue
    for a, b, d in changed_lines(r):
        rows.append({"sınıf": TR[c], "belge": Path(r.path).name, "ham OCR": a, "genel sözlük": b, "iki aşamalı": d})
exdf = pd.DataFrame(rows); exdf.to_csv(OUT / "sinif_ornekleri_sozluk.csv", index=False)
pd.set_option("display.max_colwidth", 80); exdf

In [ ]:
for c, r in picked.items():
    if r is None: continue
    lines = changed_lines(r)
    fig = plt.figure(figsize=(16, 6.2))
    img = Image.open(DATA / r.path)
    ax = fig.add_axes([0.0, 0.02, 0.24, 0.9]); ax.imshow(img, cmap="gray"); ax.axis("off")
    fig.text(0.26, 0.95, f"{TR[c]} · {Path(r.path).name}", fontsize=14, weight="bold")
    heads = ["Ham OCR", "Genel sözlük", "İki aşamalı sözlük"]
    for j, h in enumerate(heads):
        fig.text(0.26 + j * 0.25, 0.86, h, fontsize=12, weight="bold", color=["#555", "#b03a2e", "#0f6e66"][j])
    for i, trip in enumerate(lines):
        for j, s in enumerate(trip):
            fig.text(0.26 + j * 0.25, 0.78 - i * 0.12, textwrap.fill(s, 38), fontsize=9.5, family="monospace", va="top")
    fig.savefig(OUT / f"ornek_sozluk_{c}.png", dpi=150); plt.show()

## Özet

In [ ]:
print(full.to_string(index=False)); print()
print(abl[["adım", "CV şablonu", "accuracy (%)", "CV F1", "makale → CV hatası"]].to_string(index=False)); print()
print(stages.T.to_string()); print()
print("Sözlük:", info, "· alan sözlüğü:", len(domain), "kelime")
print("Çıktılar:", sorted(p.name for p in OUT.iterdir()))